# 02 · Preprocessing, chunking and the summarization pipeline

**What this does**
1. Runs the unit tests (normalization, sentence splitting, chunking, length policy, pipeline logic).
2. **Tokenizer calibration**: subword tokens per word for each candidate model's tokenizer, measured on
   XL-Sum *train* (never test). Turns word targets (10–30 % of input, 30–300 words) into decoder
   `min_new_tokens` / `max_new_tokens`. Saved to `data/processed/tokenizer_calibration.json`.
3. Measures how many benchmark texts exceed the model's direct capacity (900 / 1024 tokens).
4. Shows sentence splitting and chunking (≤ 900 tokens, ≤ 80 overlap) on one benchmark text.
5. **Pipeline smoke run** with a public XL-Sum AraBART checkpoint (baseline, not our final model) on 9 benchmark
   texts across strategies `lead`, `extractive`, `direct`, `segmentwise`, `hybrid`: output length vs. the
   10–30 % bounds, model calls and time. No ROUGE yet: human references are still being written.

Runtime: **T4 GPU** recommended (CPU works, slower). Resumable: calibration is cached on Drive.

In [ ]:
# ==== 0. Settings, pinned install, environment ==========================================
REPO_URL   = "https://github.com/assaabriiii/Offline-Arabic-Text-Summarization-Module.git"
REPO_REF   = "main"     # branch or commit SHA; use a SHA for a fully reproducible run
CONFIGS    = ["configs/default.yaml"]          # later files override earlier ones
DRIVE_ROOT = "/content/drive/MyDrive/arabic_summ"
REQUIREMENTS = "requirements/colab-data.txt"

import importlib, os, subprocess, sys
REPO_DIR = "/content/arabic_summ_repo"

def sh(cmd, secret=None):
    shown = cmd.replace(secret, "***") if secret else cmd
    print("$", shown)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stdout + r.stderr).replace(secret, "***") if secret else r.stdout + r.stderr)

token = None
try:  # private repo: add a Colab secret named GITHUB_TOKEN (key icon in the left bar)
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q {url} {REPO_DIR}", token)
sh(f"git -C {REPO_DIR} fetch -q {url} {REPO_REF} && git -C {REPO_DIR} checkout -q --detach FETCH_HEAD", token)
sh(f"pip install -q -r {REPO_DIR}/{REQUIREMENTS}")
sh(f"pip install -q --no-deps -e {REPO_DIR}")
sys.path.insert(0, f"{REPO_DIR}/src"); importlib.invalidate_caches()
os.chdir(REPO_DIR)

from arabic_summ.env import environment_report, format_report
print(format_report(environment_report()))

In [ ]:
# ==== 1. Mount Drive, load config, record environment ===================================
from google.colab import drive
drive.mount("/content/drive")

from arabic_summ.config import load_config, config_hash
from arabic_summ.env import save_environment
cfg = load_config(*CONFIGS, overrides={"paths": {"root": DRIVE_ROOT}})
save_environment(cfg, "02_preprocess")
print("artifact root:", cfg["paths"]["root"], "| config hash:", config_hash(cfg))

In [ ]:
# ==== 2. Unit tests (CPU, a few seconds) ================================================
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, "unit tests failed - stop here and paste the output"

In [ ]:
# ==== 3. Tokenizer calibration (XL-Sum train sample; cached) ============================
import pandas as pd
from transformers import AutoTokenizer
from arabic_summ import models as M
from arabic_summ.data import prepare
from arabic_summ.io_utils import read_json
from arabic_summ.text.normalize import policies_from_config
model_policy, _ = policies_from_config(cfg)
TOKENIZERS = {"arabart": "arabart", "arat5v2": "arat5v2"}   # fine-tuned baselines share these tokenizers
N_SAMPLE = 2000

cal_path = M.calibration_path(cfg)
cached = read_json(cal_path) if cal_path.exists() else {}
train = prepare.load_processed(cfg, "train").sample(N_SAMPLE, random_state=cfg["project"]["seed"])
tokenizers = {}
for key in TOKENIZERS:
    e = M.model_entry(cfg, key)
    tokenizers[key] = AutoTokenizer.from_pretrained(e["repo_id"], revision=e["revision"])
    if key in cached:
        print(f"[skip] {key}: cached"); continue
    rec = {"repo_id": e["repo_id"], "revision": e["revision"], "tokenizer_class": type(tokenizers[key]).__name__,
           "n_sample": N_SAMPLE, "sample": "xlsum train, seed 42",
           "article": M.calibrate_tokens_per_word(tokenizers[key], train.text, model_policy),
           "summary": M.calibrate_tokens_per_word(tokenizers[key], train.summary, model_policy)}
    M.save_calibration(cfg, key, rec)
cal = read_json(cal_path)
display(pd.DataFrame({k: {"tokenizer": v["tokenizer_class"], "tok/word article": v["article"]["tokens_per_word"],
                          "tok/word summary": v["summary"]["tokens_per_word"]} for k, v in cal.items()}).T)

In [ ]:
# ==== 4. How many benchmark texts exceed direct model capacity? =========================
from arabic_summ import config as C
from arabic_summ.data import benchmark as B
items = B.read_csv(C.path(cfg, "benchmark") / B.ITEMS_FILE)
items = items[items.status == "selected"] if (items.status == "selected").any() else items[items.status != "rejected"]
rows = []
for key, tok in tokenizers.items():
    n = items.text.map(lambda t: len(tok(t, add_special_tokens=False)["input_ids"]))
    for dom in sorted(items.domain.unique()):
        m = items.domain == dom
        rows.append({"tokenizer": key, "domain": dom, "texts": int(m.sum()), "tokens_median": int(n[m].median()),
                     "tokens_max": int(n[m].max()), "share_>900": round((n[m] > 900).mean(), 3),
                     "share_>1024": round((n[m] > 1024).mean(), 3)})
display(pd.DataFrame(rows))

In [ ]:
# ==== 5. Sentence splitting + chunking on the longest benchmark text ====================
from arabic_summ.text.chunk import chunk_sentences
from arabic_summ.text.normalize import normalize
from arabic_summ.text.segment import split_sentences
tok = tokenizers["arabart"]
count = lambda t: len(tok(t, add_special_tokens=False)["input_ids"])
longest = items.loc[items.word_count.astype(int).idxmax()]
sents = split_sentences(normalize(longest.text, model_policy))
print(longest.sample_id, longest.word_count, "words |", len(sents), "sentences")
for name, mx in [("direct (900/80)", 900), ("segmentwise", cfg["pipeline"]["segmentwise"]["segment_max_tokens"])]:
    ch = chunk_sentences(sents, count, mx, cfg["pipeline"]["chunking"]["overlap_tokens"],
                         cfg["pipeline"]["chunking"]["paragraph_break_min_fill"])
    print(f"{name:18s}: {len(ch)} chunks, tokens={[c.n_tokens for c in ch]}, overlap sentences={[c.n_overlap_sentences for c in ch]}")
print("\nfirst 5 sentences:"); [print("  -", s.text[:120]) for s in sents[:5]]

In [ ]:
# ==== 6. Pipeline smoke run with a public XL-Sum baseline ================================
import json, time, torch
from arabic_summ.summarize.pipeline import Summarizer
BASELINE = "arabart_xlsum_omar"            # see configs/default.yaml -> models.baselines
STRATEGIES = ["lead", "extractive", "direct", "segmentwise", "hybrid"]
RATIO = cfg["pipeline"]["default_ratio"]

backend = M.load_hf_backend(cfg, BASELINE, dtype="float16" if torch.cuda.is_available() else "float32")
summ = Summarizer(cfg, backend=backend, tokens_per_word=M.tokens_per_word(cfg, "arabart"))
print(backend.name, "| device:", backend.device, "| tokens/word:", summ.tokens_per_word)

demo = (items.groupby(["domain", "length_bin"], group_keys=False)
             .apply(lambda g: g.sample(min(len(g), 2), random_state=0)).head(9))
out_dir = C.path(cfg, "logs") / "02_pipeline_demo"; out_dir.mkdir(parents=True, exist_ok=True)
records = []
for _, r in demo.iterrows():
    for st in STRATEGIES:
        res = summ.summarize(r.text, ratio=RATIO, strategy=st)
        records.append({"sample_id": r.sample_id, "domain": r.domain, **{k: v for k, v in res.to_dict().items() if k != "parts"}})
pd.DataFrame(records).to_json(out_dir / "results.jsonl", orient="records", lines=True, force_ascii=False)
df = pd.DataFrame(records)
display(df.groupby("strategy").agg(length_ok=("length_ok", "mean"), words=("output_words", "median"),
                                   target=("target_words", "median"), calls=("n_model_calls", "mean"),
                                   seconds=("seconds", lambda s: round(pd.Series([x["total"] for x in s]).median(), 2))))

In [ ]:
# ==== 7. Read one example per strategy (paste this back) ================================
ex = demo.iloc[0]
print(ex.sample_id, ex.domain, ex.word_count, "words\n")
for st in STRATEGIES:
    r = next(x for x in records if x["sample_id"] == ex.sample_id and x["strategy"] == st)
    print(f"--- {st}: {r['output_words']} words (bounds {r['bounds']}), {r['seconds']['total']}s\n{r['summary']}\n")